# Sentinel-2 Data Availability Across Catalogues

The same Sentinel-2 archive is served by several STAC catalogues, and they do not all list the same scenes for the same area. Before committing to a long download, it is worth checking which one covers your area best.

`check_scene_availability` queries every catalogue that serves the mission and returns a per-day table of scene counts.

| catalogue | key | notes |
|---|---|---|
| Element84 (Earth Search) | `element84` | the default source in stac2cube |
| Planetary Computer | `planetary_computer` | L2A only, no L1C collection |
| terrabyte (DLR / LRZ) | `terrabyte` | L1C is listed here for comparison, but downloading it needs a request to DLR |
| Copernicus Data Space | `cdse` | catalogue search is anonymous, reading pixels needs free access keys in `credentials/cdse_key` |

> **What this measures.** Raw catalogue listing. No cloud filter is applied, and a scene being listed does not mean its pixels can be downloaded anonymously. A catalogue that fails, for example because its keys are missing or the API times out, comes back as an all-empty column rather than aborting the whole table.

---

In [ ]:
from stac2cube import check_scene_availability, satellite_map
import pandas as pd

pd.set_option("display.max_rows", 200)

## 1. Choose the area

Use an existing polygon file, or draw one on the map below.

In [ ]:
polygon = "../polygons/test.gpkg"     # file, or a WGS84 bbox [xmin, ymin, xmax, ymax]

**Optional: draw it instead.** Use the drawing tools at the top left, then run the cell after the map.

In [ ]:
m = satellite_map(center=(41.042, 29.017), zoom=11, draw=True)
m

In [ ]:
if m is not None and m.user_roi is not None:
    geom = m.user_roi["geometry"]
    ring = geom["coordinates"][0]
    xs = [p[0] for p in ring]
    ys = [p[1] for p in ring]
    polygon = [min(xs), min(ys), max(xs), max(ys)]
    print("Using the drawn area:", polygon)
else:
    print("Nothing drawn, keeping:", polygon)

## 2. Sentinel-2 L2A

`daterange` takes the same forms as the builder: a window, a season repeating over years, or `None` for the entire archive.

> A large area over the full archive means a lot of paging and can take several minutes.

In [ ]:
df_l2a, errors_l2a = check_scene_availability(
    mission="sentinel_2_l2a",
    polygon=polygon,
    daterange=["2024-04-01", "2024-06-30"],
)
df_l2a.head(20)

In [ ]:
for label, msg in errors_l2a.items():
    print(f"{label}: {msg}")

### Totals per catalogue

Each column counts scenes; the number of acquisition **days** is usually the more useful figure, because one day can be delivered as several tiles.

In [ ]:
summary_l2a = pd.DataFrame({
    "scenes": df_l2a.sum(),
    "days_with_a_scene": (df_l2a > 0).sum(),
})
summary_l2a

### Days one catalogue has and another does not

The catalogue with the most acquisition days is taken as the reference, and the rest are compared against it.

In [ ]:
def missing_vs_best(df):
    days = {c: set(df.index[df[c].fillna(0) > 0]) for c in df.columns if df[c].notna().any()}
    if not days:
        return pd.DataFrame(), None, set()
    best = max(days, key=lambda c: len(days[c]))
    rows = []
    for c in df.columns:
        if c not in days:
            rows.append({"catalogue": c, "days": None, "missing_vs_best": None})
            continue
        rows.append({
            "catalogue": c,
            "days": len(days[c]),
            "missing_vs_best": len(days[best] - days[c]),
        })
    return pd.DataFrame(rows).set_index("catalogue"), best, days

table, best, days = missing_vs_best(df_l2a)
print("reference (most complete):", best)
table

### Which dates exactly

In [ ]:
if best:
    for c, d in days.items():
        miss = sorted(days[best] - d)
        print(f"{c}: missing {len(miss)} day(s)")
        if miss:
            print("   " + ", ".join(miss))

## 3. Sentinel-2 L1C

L1C matters for cloud masking: the s2cloudless model in `2_Cloudmask_Data_Cube.ipynb` runs on L1C scenes. Element84 is the only catalogue stac2cube can actually download L1C pixels from, and its L1C archive is sparse before roughly 2021. If the table below shows L1C gaps for your period, expect `get_cloud_layers` to fall back to the SCL mask on those dates, which is what `missing_l1c="scl"` does.

In [ ]:
df_l1c, errors_l1c = check_scene_availability(
    mission="sentinel_2_l1c",
    polygon=polygon,
    daterange=["2024-04-01", "2024-06-30"],
)
df_l1c.head(20)

In [ ]:
for label, msg in errors_l1c.items():
    print(f"{label}: {msg}")

In [ ]:
pd.DataFrame({
    "scenes": df_l1c.sum(),
    "days_with_a_scene": (df_l1c > 0).sum(),
})

## 4. L2A dates without an L1C scene

These are the dates a cube would hold but s2cloudless could not be run on directly.

In [ ]:
l2a_days = set(df_l2a.index[df_l2a.fillna(0).sum(axis=1) > 0]) if len(df_l2a) else set()
e84_l1c = "Element84"
if e84_l1c in df_l1c.columns:
    l1c_days = set(df_l1c.index[df_l1c[e84_l1c].fillna(0) > 0])
    gap = sorted(l2a_days - l1c_days)
    print(f"{len(gap)} L2A day(s) with no Element84 L1C scene")
    if gap:
        print(", ".join(gap))
else:
    print("Element84 L1C column not available, cannot compare.")

## 5. Same check inside the interface

The Data Cube Builder in `interactive/User_Interface_Tools.ipynb` runs exactly this check behind its "Check data availability" button, using the area and date range already set for the cube.